## **2 кейс**

**Выгрузка активности с ItResume**

**Важно**

Перед началом решения выполните следующую ячейку, чтобы загрузить необходимый для работы файл.

In [1]:
!wget https://gist.github.com/Vs8th/a7a7f00e6cdef1b3fe87e4d61ca56e5f/raw/codesubmit.csv

--2026-03-13 09:33:41--  https://gist.github.com/Vs8th/a7a7f00e6cdef1b3fe87e4d61ca56e5f/raw/codesubmit.csv
Resolving gist.github.com (gist.github.com)... 140.82.112.4
Connecting to gist.github.com (gist.github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/a7a7f00e6cdef1b3fe87e4d61ca56e5f/raw/codesubmit.csv [following]
--2026-03-13 09:33:41--  https://gist.githubusercontent.com/Vs8th/a7a7f00e6cdef1b3fe87e4d61ca56e5f/raw/codesubmit.csv
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 215378 (210K) [text/plain]
Saving to: ‘codesubmit.csv’

codesubmit.csv      100%[===================>] 210.33K  --.-KB/s    in 0.03s   

2026-03-13 09:33:41 (8.04 MB/s) - ‘co

Чтобы посмотреть как он выглядит выполните следующую ячейку.

In [48]:
import pandas as pd

df = pd.read_csv('codesubmit.csv', sep = ';', parse_dates=['created_at'])
df

,created_at,user_id,problem_id,is_correct,type
0,2023-04-30 13:47:14.344471,7,870,1.0,submit
1,2023-04-30 13:46:15.949925,7,870,0.0,submit
2,2023-04-30 16:13:26.005286,173,21,1.0,submit
3,2023-04-30 16:13:06.739782,173,21,NaN,run
4,2023-04-30 15:52:00.195532,173,25,1.0,submit
...,...,...,...,...,...
4994,2023-04-30 21:52:00.269123,13493,435,NaN,run
4995,2023-04-30 21:51:01.094234,13493,435,1.0,submit
4996,2023-04-30 21:50:52.059690,13493,435,NaN,run
4997,2023-04-30 21:42:24.323689,13493,1086,NaN,run


### **Решения**

#### **Задача 1**

Ваша задача - выяснить сколько в среднем тратится времени на решение задачи.

**Примечание**: для правильного подсчета - рассчитайте сначала среднее время решения по каждой задаче в отдельности, и только затем находите общее среднее время решения задач.

Результат - число типа `float`, округлите до 2 знаков после запятой и запишите в переменную `res`.


**Решение**

Напишите свое решение ниже

In [3]:
df = df.sort_values(['user_id', 'problem_id', 'created_at']).reset_index(drop=True)
grouped = df.groupby(['user_id', 'problem_id'])

first_attempt = grouped['created_at'].min().reset_index(name='first_attempt')

first_success = (
    df[(df['type'] == 'submit') & (df['is_correct'] == 1)]
    .groupby(['user_id', 'problem_id'])['created_at']
    .min()
    .reset_index(name='first_success')
)

result = first_attempt.merge(first_success, on=['user_id', 'problem_id'], how='inner')
first_attempt = (
    df.groupby(['user_id','problem_id'])['created_at']
    .min()
    .reset_index(name='first_attempt')
)
first_success = (
    df[(df['type']=='submit') & (df['is_correct']==1)]
    .groupby(['user_id','problem_id'])['created_at']
    .min()
    .reset_index(name='first_success')
)
result = first_attempt.merge(
    first_success,
    on=['user_id', 'problem_id'],
    how='inner'
)
result['solve_time'] = result['first_success'] - result['first_attempt']
result = result[result['solve_time'] > pd.Timedelta(0)]
result['solve_seconds'] = result['solve_time'].dt.total_seconds()

problem_mean = (
    result.groupby('problem_id')['solve_seconds']
    .mean()
)
res = round(problem_mean.mean(), 2)

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [4]:
try:
    assert res == 611.86
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


#### **Задача 2**

Ваша задача - выяснить сколько часов в среднем проводит юзер в день на платформе. Перерывы в активности за день - не учитываем.

Результат - число типа `float`, округлите до 2 знаков после запятой и запишите в переменную `res2`.

**Решение**

Напишите свое решение ниже

In [39]:

# Ваше решение здесь - решение без учета перерывов в активности
df = df.sort_values(['user_id', 'created_at']).reset_index(drop=True)
df['time_diff'] = df.groupby('user_id')['created_at'].diff()
df['new_session']=(df['time_diff']>pd.Timedelta(hours=1)).astype(int)
df['session_id'] = df.groupby('user_id')['new_session'].cumsum()
df[['user_id','created_at','time_diff','new_session','session_id']].head(20)
sessions = (df.groupby(['user_id', 'session_id']).agg(start_time=('created_at','min'), end_time = ('created_at','max')).reset_index())
sessions['session_duration'] = (sessions['end_time'] - sessions['start_time']).dt.total_seconds()
sessions['date'] = sessions['start_time'].dt.date
daily_time = (sessions.groupby(['user_id', 'date'])['session_duration'].sum().reset_index(name='daily_time'))
res2 = round(daily_time['daily_time'].mean()/3600, 2)
res2
#daily_time.head(10)
#res2 =

np.float64(0.97)

In [40]:
df = pd.read_csv('codesubmit.csv', sep=';', parse_dates=['created_at'])

df = df.sort_values(['user_id', 'created_at']).reset_index(drop=True)

df['time_diff'] = df.groupby('user_id')['created_at'].diff()

user_time = (
    df.groupby('user_id')['time_diff']
    .sum()
    .dt.total_seconds()
    .div(3600)
)

res2 = round(user_time.mean(), 2)
res2

np.float64(1.7)

In [36]:
from numpy import average
import csv
from collections import defaultdict
import datetime
filename = 'codesubmit.csv'
rows = []
with open(filename, 'r') as csvfile:
    csvreader = csv.DictReader(csvfile, delimiter=';')
    for row in csvreader:
        rows.append(row)
user_attempts = {}
for row in rows:
    user_id = row['user_id']
    if user_id not in user_attempts:
        user_attempts[user_id] = []
    user_attempts[user_id].append(row)
    user_attempts[user_id].sort(key=lambda x: datetime.datetime.strptime(x['created_at'], '%Y-%m-%d %H:%M:%S.%f'))
user_time = {}
for user_id, attempts in user_attempts.items():
  daily_time = 0
  day_start = None
  for attemp in attempts:
    created_at = datetime.datetime.strptime(attemp['created_at'], '%Y-%m-%d %H:%M:%S.%f')
    if day_start == None:
      day_start = created_at
    daily_time += (created_at - day_start).total_seconds()
    day_start = created_at
  user_time[user_id] = daily_time/3600
average_time = average(list(user_time.values()))
res2 = round(average_time, 2)
res2

np.float64(1.7)

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [41]:
try:
    assert res2 == 1.7
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


#### **Задача 3**

Теперь давайте посмотрим на активные сеансы. Выясните, сколько задач в среднем решается за один активный сеанс.

**Активный сеанс** - период, когда между любой активностью пользователя разница менее или равна часу, не более

**Важно**: в расчет берем не только успешные попытки решений (`is_correct=1`), а и неуспешные тоже (`is_correct=0`), и тип `run` в том числе.

Результат - число типа `float`, округлите до 2 знаков после запятой и запишите в переменную `res3`.

**Решение**

Напишите свое решение ниже

In [46]:

# Ваше решение здесь
df = df.sort_values(['user_id', 'created_at']).reset_index(drop=True)
df['time_diff'] = df.groupby('user_id')['created_at'].diff()
df['new_session']=(df['time_diff']>pd.Timedelta(hours=1)).astype(int)
df['session_id'] = df.groupby('user_id')['new_session'].cumsum()
session_tasks = (df.groupby(['user_id', 'session_id'])['problem_id'].nunique().reset_index(name='tasks_per_session'))
res3 = round(session_tasks['tasks_per_session'].mean(), 2)
res3

np.float64(3.14)

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [47]:
try:
    assert res3 == 3.14
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


#### **Задача 4**

И финальная - найдите самый "популярный" час дня на нашей платформе.

Популярность определяем максимальным количеством уникальных пользователей, совершающих какую-либо активность в этот период

Результат в числовом формате запишите в переменную `res4`.

Например, самым популярным часом стал период с 22 до 23, тогда в переменной `res4` должно лежать **22**. Обозначающее начало этого периода.

**Решение**

Напишите свое решение ниже

In [52]:

# Ваше решение здесь
df['hour'] = df['created_at'].dt.hour
hours_user = df.groupby('hour')['user_id'].nunique()
res4 = hours_user.idxmax()


#res4 =

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [53]:
try:
    assert res4 == 16
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!
